# Garden Store Agent

A conversational agent that uses the Garden Store A2A interface via a local Ollama model.

**Before running:**
- Garden store running: `docker compose up`
- Ollama running with a model pulled: `ollama pull qwen2.5:7b`

Run cells 1–3 once to initialise, then use the chat widget in cell 4.

In [ ]:
import json

import httpx
import ipywidgets as widgets
from IPython.display import display
from openai import OpenAI

# ---------------------------------------------------------------------------
# Configuration — adjust if Ollama or the store are on a different host
# ---------------------------------------------------------------------------
OLLAMA_BASE  = "http://localhost:11434"  # Windows host from WSL: use host IP
OLLAMA_MODEL = "llama3.1:8b"
GARDEN_STORE = "http://localhost:8000"


In [ ]:

# ---------------------------------------------------------------------------
# A2A helpers
# ---------------------------------------------------------------------------

def fetch_card(store_base: str) -> dict:
    r = httpx.get(f"{store_base}/.well-known/agent-card.json", timeout=10)
    r.raise_for_status()
    return r.json()


def build_system_prompt(card: dict, authenticated: bool) -> str:
    skills = card.get("skills", [])
    skills_text = "\n\n".join(
        "Action: {name}\n{description}\nExample intent: {example}".format(
            name=s["name"],
            description=s["description"],
            example=s.get("examples", ["(none)"])[0],
        )
        for s in skills
    )
    auth_note = (
        "The user is authenticated — checkout and list_purchases are available."
        if authenticated else
        "The user is NOT authenticated — browsing only. "
        "If the user tries to checkout or view purchases, tell them to log in first."
    )
    return (
        f"You are a helpful assistant for {card['name']}.\n\n"
        f"{card['description']}\n\n"
        f"## Available actions\n\n{skills_text}\n\n"
        f"## Authentication status\n\n{auth_note}\n\n"
        "## Rules you must follow\n\n"
        "0. ALWAYS respond in English, regardless of any other language you detect.\n"
        "1. If you need data from the store, call the tool NOW — do not say you will call it, "
        "do not explain what you are about to do, just call it immediately.\n"
        "2. NEVER invent or guess a product_id. "
        "Always call browse_products first to find the correct product_id before checkout.\n"
        "3. NEVER describe placing an order or confirm a result unless a tool call was actually made.\n"
        "4. Once the user confirms checkout, call the tool immediately — do not narrate, just act.\n"
        "5. Summarise tool results in plain English. Do not show raw JSON to the user.\n"
        "6. If a tool returns {\"error\": \"unauthorized\"}, tell the user they need to log in.\n"
        "7. You may combine tool results with your own general knowledge. For example: call "
        "browse_products to see what is available, then use your own knowledge to answer "
        "questions like which products suit a season, a climate, or a skill level."
    )


_rpc_seq = 0

def a2a_call(store_base: str, intent: dict, token: str | None = None) -> dict:
    global _rpc_seq
    _rpc_seq += 1
    headers = {"Content-Type": "application/json", "A2A-Version": "1.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    payload = {
        "jsonrpc": "2.0",
        "id": f"nb-{_rpc_seq}",
        "method": "SendMessage",
        "params": {
            "message": {
                "role": "ROLE_USER",
                "parts": [{"text": json.dumps(intent)}],
                "messageId": f"msg-{_rpc_seq}",
            }
        },
    }
    r = httpx.post(
        f"{store_base}/rpc",
        json=payload,
        headers=headers,
        timeout=30,
    )
    r.raise_for_status()
    envelope = r.json()
    if "error" in envelope:
        return {"error": envelope["error"]}
    text = envelope["result"]["message"]["parts"][0]["text"]
    return json.loads(text)


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "call_agent",
            "description": (
                "Send a JSON intent to the agent and receive a response. "
                "Construct the intent object exactly as documented in the available actions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "intent": {
                        "type": "object",
                        "description": 'A JSON object with an "action" field and parameters as documented.',
                    }
                },
                "required": ["intent"],
            },
        },
    }
]


In [ ]:

# ---------------------------------------------------------------------------
# Initialise — run once; re-run to reset the conversation
# ---------------------------------------------------------------------------

card          = fetch_card(GARDEN_STORE)
llm           = OpenAI(base_url=f"{OLLAMA_BASE}/v1", api_key="ollama")

# _token and _customer are set by the login cell above (None if skipped)
_authenticated = '_token' in dir() and _token is not None
system_prompt  = build_system_prompt(card, authenticated=_authenticated)
messages       = [{"role": "system", "content": system_prompt}]

print(f"Connected to : {card['name']}")
print(f"Model        : {OLLAMA_MODEL}")
print(f"Auth         : {'authenticated as ' + _customer['name'] if _authenticated else 'not authenticated (browse only)'}")
print(f"Skills       : {', '.join(s['name'] for s in card.get('skills', []))}")


In [ ]:

# ---------------------------------------------------------------------------
# Login (optional) — run this cell to authenticate before using the chat.
# Option A: paste a developer token from the Account page on the website.
# Option B: enter email + password.
# Skip entirely to browse without logging in.
# ---------------------------------------------------------------------------
import getpass as _gp

_token    = None
_customer = None

_dev_token = input("Developer token (Enter to skip / use email+password): ").strip()

if _dev_token:
    _token = _dev_token
    # Decode the payload to show who we're acting as (no signature check needed here)
    import base64 as _b64, json as _json
    try:
        _payload = _json.loads(_b64.b64decode(_dev_token.split(".")[1] + "=="))
        _customer = {"name": _payload.get("email", "unknown"), "email": _payload.get("email", "")}
        print(f"Using developer token for {_customer['email']}")
    except Exception:
        print("Using developer token (could not decode payload).")
else:
    _email = input("Email (Enter to skip): ").strip()
    if _email:
        _password = _gp.getpass("Password: ")
        import httpx as _httpx
        _resp = _httpx.post(f"{GARDEN_STORE}/auth/login", json={"email": _email, "password": _password}, timeout=10)
        if _resp.status_code == 401:
            print("Invalid credentials — continuing without authentication.")
        else:
            _resp.raise_for_status()
            _token    = _resp.json()["access_token"]
            _customer = _resp.json()["customer"]
            print(f"Signed in as {_customer['name']} ({_customer['email']})")
    else:
        print("Skipped — browsing only.")


In [ ]:

# ---------------------------------------------------------------------------
# Chat UI
# ---------------------------------------------------------------------------

CSS = """
<style>
.msg-you   { background:#e8f4e8; border-left:3px solid #3a5c2c; padding:8px 12px; margin:4px 0; border-radius:4px; color:#1a3a10; }
.msg-agent { background:#f0f7ff; border-left:3px solid #2c5c8c; padding:8px 12px; margin:4px 0; border-radius:4px; color:#0d2a4a; }
.msg-tool  { background:#fafafa; border-left:3px solid #aaa;    padding:6px 12px; margin:2px 0; border-radius:4px;
             font-family:monospace; font-size:0.85em; color:#444; }
.msg-error { background:#fff0f0; border-left:3px solid #c0392b; padding:8px 12px; margin:4px 0; border-radius:4px; color:#c0392b; }
</style>
"""

out        = widgets.Output(layout=widgets.Layout(
                border='1px solid #dce8cc',
                border_radius='8px',
                min_height='300px',
                padding='8px',
            ))
text_input = widgets.Text(placeholder='Type your request and press Enter…',
                          layout=widgets.Layout(flex='1'))
send_btn   = widgets.Button(description='Send',       button_style='success')
clear_btn  = widgets.Button(description='Clear chat', button_style='warning')

_auth_label = f"Signed in as {_customer['name']}" if _authenticated else "Not signed in (browse only)"
with out:
    display(widgets.HTML(CSS + f'<p style="color:#555;font-style:italic">{_auth_label} — {card["name"]}.</p>'))


def _html(cls, content):
    return widgets.HTML(f'<div class="{cls}">{content}</div>')


def on_send(_):
    user_text = text_input.value.strip()
    if not user_text:
        return

    text_input.value = ''
    send_btn.disabled = True
    send_btn.description = 'Thinking…'

    with out:
        display(_html('msg-you', f'<b>You:</b> {user_text}'))

    messages.append({"role": "user", "content": user_text})

    try:
        while True:
            response = llm.chat.completions.create(
                model=OLLAMA_MODEL,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                with out:
                    display(_html('msg-agent', f'<b>Agent:</b> {msg.content}'))
                break

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                intent = args.get("intent", args)
                # Some models double-encode intent as a JSON string rather than an object
                if isinstance(intent, str):
                    try:
                        intent = json.loads(intent)
                    except json.JSONDecodeError:
                        pass
                result = a2a_call(GARDEN_STORE, intent, _token)
                summary = json.dumps(result)

                with out:
                    display(_html('msg-tool',
                        f'→ {json.dumps(intent)}<br>'
                        f'← {summary[:200] + "…" if len(summary) > 200 else summary}'
                    ))

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                })

    except Exception as e:
        with out:
            display(_html('msg-error', f'<b>Error:</b> {e}'))
    finally:
        send_btn.disabled = False
        send_btn.description = 'Send'


def on_clear(_):
    global messages
    messages = [{"role": "system", "content": system_prompt}]
    out.clear_output()
    with out:
        display(widgets.HTML(CSS + f'<p style="color:#555;font-style:italic">Chat cleared. {_auth_label}.</p>'))


send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)

display(widgets.VBox([
    out,
    widgets.HBox([text_input, send_btn, clear_btn],
                 layout=widgets.Layout(margin='8px 0 0 0', gap='6px'))
]))
